In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import joblib

print("🔄 स्टेप 1: डेटा लोड हो रहा है...")
df = pd.read_csv('Final_Prepared_Data.csv')

print("🛠️ स्टेप 2: प्रोजेक्ट रिक्वायरमेंट के अनुसार Feature Engineering की जा रही है...")
# 1. Balance-to-Salary Ratio
df['BalanceToSalaryRatio'] = df['Balance'] / (df['EstimatedSalary'] + 1)
# 2. Product Density Indicator
df['ProductDensity'] = df['NumOfProducts'] / (df['Tenure'] + 1)
# 3. Age-Tenure Interaction
df['AgeTenureInteraction'] = df['Age'] * df['Tenure']

# Features और Target अलग करना
X = df.drop(columns=['Exited'])
y = df['Exited']

# ट्रेन-टेस्ट स्प्लिट
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("⚖️ स्टेप 3: SMOTE लागू किया जा रहा है...")
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("🤖 स्टेप 4: एडवांस XGBoost MODEL ट्रेन हो रहा है...")
model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
model.fit(X_train_res, y_train_res)

print("🎯  स्टेप 5: प्रेडिक्टिव रिस्क स्कोर जनरेट हो रहा है...")
df['Churn_Probability'] = model.predict_proba(X)[:, 1]
df['ML_Risk_Score'] = (df['Churn_Probability'] * 100).astype(int)

df['Risk_Tier'] = pd.cut(df['ML_Risk_Score'], bins=[-1, 30, 60, 100], labels=['Low Risk', 'Medium Risk', 'High Risk'])

print("💾  स्टेप 6: नई फाइल सेव हो रही है...")
df.to_csv('Bank_Churn_With_ML_Scores.csv', index=False)

# इस मॉडल को भविष्य के लिए सेव करना (Streamlit Web App के लिए ज़रूरी)
joblib.dump(model, 'xgb_churn_model.pkl')
joblib.dump(X.columns.tolist(), 'model_features.pkl')

print("✅ सफलता! मॉडल और फ़ीचर इंजीनियरिंग दोनों अपडेट हो चुके हैं।")


🔄 स्टेप 1: डेटा लोड हो रहा है...
🛠️ स्टेप 2: प्रोजेक्ट रिक्वायरमेंट के अनुसार Feature Engineering की जा रही है...
⚖️ स्टेप 3: SMOTE लागू किया जा रहा है...
🤖 स्टेप 4: एडवांस XGBoost MODEL ट्रेन हो रहा है...
🎯  स्टेप 5: प्रेडिक्टिव रिस्क स्कोर जनरेट हो रहा है...
💾  स्टेप 6: नई फाइल सेव हो रही है...
✅ सफलता! मॉडल और फ़ीचर इंजीनियरिंग दोनों अपडेट हो चुके हैं।
